# GLiNER2.5-Decide: SOC Multi-Head Operational Classification & Routing

This notebook demonstrates **GLiNER2.5-Decide** (Fastino, 340M DeBERTa-v3-large encoder) applied to **Security Operations Center (SOC) operational decisions**.

Unlike classic GLiNER (which extracts token-level entity spans like CVEs and malware names), **GLiNER2.5-Decide** evaluates arbitrary multi-head zero-shot classification schemas in a single forward pass (~10–30 ms on CPU/GPU) without prompt templates or token generation.

### Use Cases Covered:
1. **Tier 1 Alert Triage:** Incident severity, urgency tier (P1/P2/P3), and disposition hypothesis.
2. **Specialist Sub-Agent Routing:** Dispatch to Tier 2 Investigator, Threat Hunter, CTI Researcher, or Detection Engineer.
3. **Human-in-the-Loop (HITL) Policy Gatekeeping:** Authorization requirements and investigation completion status.
4. **CTI Feed Ingestion Filtering:** Cloud enterprise relevance and threat categorization.
5. **Latency & Throughput Benchmark:** Sub-50ms CPU edge execution metrics.

In [ ]:
# Install gliner2 with local PyTorch runtime support
!pip install -q "gliner2[local]"

In [ ]:
import time
import json
import torch
from gliner2 import AutoExtractor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on device: {DEVICE}")

t0 = time.perf_counter()
model = AutoExtractor.from_pretrained("fastino/GLiNER2.5-Decide", map_location=DEVICE)
print(f"Model loaded in {time.perf_counter() - t0:.2f}s")

def decide(text: str, schema: dict, **kwargs) -> dict:
    t0 = time.perf_counter()
    result = model.classify_text(text, schema, **kwargs)
    ms = (time.perf_counter() - t0) * 1000
    print(f"INPUT: {text[:120]}...")
    print(f"OUTPUT ({ms:.1f} ms):")
    print(json.dumps(result, indent=2))
    print("-" * 60)
    return result

## 1. Tier 1 Alert Triage & Disposition Hypothesis
Score severity, urgency tier, and initial disposition in a single forward pass.

In [ ]:
ALERT_TRIAGE_SCHEMA = {
    "severity": ["critical", "high", "medium", "low", "informational"],
    "urgency_tier": ["p1_immediate_containment", "p2_same_shift_triage", "p3_investigative_backlog"],
    "disposition_hypothesis": [
        "true_positive_malicious",
        "benign_scanner_traffic",
        "rfc5737_test_artifact",
        "known_false_positive"
    ]
}

# Test Case 1: Active Ransomware / Beaconing
alert_1 = "High frequency outbound beaconing observed from Domain Controller DC-01 to 198.51.100.44 with Mimikatz memory dump commands."
decide(alert_1, ALERT_TRIAGE_SCHEMA)

# Test Case 2: RFC-5737 Benchmark Artifact
alert_2 = "Automated pipeline simulated test attack originating from RFC-5737 documentation prefix 192.0.2.15 for compliance testing."
decide(alert_2, ALERT_TRIAGE_SCHEMA)

# Test Case 3: Authorized Security Scanner
alert_3 = "Routine Qualys vulnerability scanner IP 10.0.4.12 executing HTTP GET probes against internal staging web service."
decide(alert_3, ALERT_TRIAGE_SCHEMA)

## 2. Specialist Sub-Agent Routing
Dispatch incidents directly to the correct agent specialist without multi-second orchestrator reasoning.

In [ ]:
SPECIALIST_ROUTING_SCHEMA = {
    "assigned_specialist": [
        "tier2_investigator",
        "threat_hunter",
        "cti_researcher",
        "detection_engineer"
    ]
}

# Case A: Complex lateral movement across subnets
case_a = "Compromised workstation pivoted to internal file server via SMB PsExec with ticket-granting ticket forgery."
decide(case_a, SPECIALIST_ROUTING_SCHEMA)

# Case B: Periodic statistical anomaly / beaconing
case_b = "DNS traffic analysis detected low-frequency queries with high Shannon entropy indicative of DGA tunneling across edge VPC."
decide(case_b, SPECIALIST_ROUTING_SCHEMA)

# Case C: Unstructured CISA advisory
case_c = "CISA Alert AA24-210A published detailing Volt Typhoon exploitation of zero-day appliances and living-off-the-land binaries."
decide(case_c, SPECIALIST_ROUTING_SCHEMA)

# Case D: False positive rule tuning
case_d = "Chronicle YARA-L rule Multiple_Failed_Logins generating 500 alerts per hour due to scheduled service account password sync."
decide(case_d, SPECIALIST_ROUTING_SCHEMA)

## 3. Human-in-the-Loop (HITL) Policy Gatekeeping
Enforce organizational safety policies before executing high-impact containment or closing cases.

In [ ]:
HITL_POLICY_GATE_SCHEMA = {
    "requires_human_approval": [
        "requires_soc_manager_approval",
        "safe_autonomous_investigation"
    ],
    "agent_completion_status": [
        "investigation_complete",
        "insufficient_evidence_pivot_required",
        "blocked_awaiting_telemetry"
    ]
}

action_1 = "Revoke all OAuth access tokens and reset passwords for executive account ceo@corp.com after suspicious login."
decide(action_1, HITL_POLICY_GATE_SCHEMA)

action_2 = "Query VirusTotal and Chronicle UDM events for IP 198.51.100.22 over past 7 days."
decide(action_2, HITL_POLICY_GATE_SCHEMA)

action_3 = "Completed root cause analysis, confirmed false positive scanner traffic, documented summary in case notes."
decide(action_3, HITL_POLICY_GATE_SCHEMA)

## 4. Unstructured CTI Advisory Ingestion Gating
Pre-screen security advisories to filter out non-applicable platform bulletins.

In [ ]:
CTI_APPLICABILITY_SCHEMA = {
    "enterprise_relevance": [
        "actionable_threat_advisory",
        "unrelated_vendor_patch",
        "general_awareness"
    ],
    "target_platform": [
        "gcp_cloud",
        "kubernetes",
        "windows_ad",
        "linux_endpoints",
        "saas_identity"
    ]
}

cisa_text = "CISA advisory warns of privilege escalation vulnerability in Google Kubernetes Engine (GKE) clusters running on GCP with Anthos."
decide(cisa_text, CTI_APPLICABILITY_SCHEMA)

## 5. Latency & Performance Benchmark
Evaluate single-pass operational decision speed on CPU/GPU.

In [ ]:
num_runs = 20
durations = []

sample_text = "Outbound beaconing to suspected C2 server detected on Linux production server node-04."
print(f"Benchmarking {num_runs} forward passes on {DEVICE}...")

for _ in range(num_runs):
    t0 = time.perf_counter()
    _ = model.classify_text(sample_text, ALERT_TRIAGE_SCHEMA)
    durations.append((time.perf_counter() - t0) * 1000)

durations.sort()
mean_lat = sum(durations) / len(durations)
median_lat = durations[len(durations) // 2]
p95_lat = durations[int(len(durations) * 0.95)]

print(f"Mean Latency   : {mean_lat:.2f} ms")
print(f"Median Latency : {median_lat:.2f} ms")
print(f"P95 Latency    : {p95_lat:.2f} ms")